# 03 — Auditoría de datos para elección modal

Este notebook audita las cinco tablas de la SQLite relacional en modo solo lectura. Valida relaciones y cobertura, perfila los campos y genera un diccionario analítico separado. No crea aún una tabla enriquecida ni estima modelos. Los valores fuente no se recodifican; costos, pagos y tiempos de etapas se revisan sin agregarlos al viaje.

Los reportes contienen métricas agregadas, sin identificadores ni filas de microdatos. Ejecutar desde la raíz del proyecto o desde este directorio.

In [ ]:
from pathlib import Path
import pandas as pd
from mode_choice.mode_choice_data_audit import audit_mode_choice_data

PROJECT_ROOT = next((candidate for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (candidate / 'pyproject.toml').is_file()), None)
assert PROJECT_ROOT is not None, 'Could not locate the project root.'
result = audit_mode_choice_data(PROJECT_ROOT)
result

In [ ]:
relation_audit = pd.read_csv(PROJECT_ROOT / result['outputs']['quality'], dtype=str, keep_default_na=False)
field_profile = pd.read_csv(PROJECT_ROOT / result['outputs']['profiles'], dtype=str, keep_default_na=False)
duplicate_comparison = pd.read_csv(PROJECT_ROOT / result['outputs']['comparisons'], dtype=str, keep_default_na=False)
dictionary = pd.read_csv(PROJECT_ROOT / result['outputs']['dictionary'], dtype=str, keep_default_na=False)

assert result['table_counts'] == {'viajes': 100174, 'personas': 67556, 'hogares': 22755, 'vehiculos': 17392, 'etapas': 110881}
assert result['trip_person_matches'] == 100174
assert result['trip_household_matches'] == 100174
assert result['trips_without_stages'] == 0
assert result['trips_with_multiple_stages'] == 9492
assert len(dictionary) == result['dictionary_rows'] == result['profile_rows']
assert not any(column in {'key_viaje', 'key_hg', 'key_persona', 'cod_per', 'cod_pers'} for frame in [relation_audit, field_profile, duplicate_comparison] for column in frame.columns)
print({key: value for key, value in result.items() if key != 'outputs'})
display(relation_audit)
display(duplicate_comparison)
display(field_profile.head(20))
display(dictionary.head(20))

## Lectura de resultados

Las tablas de relación distinguen llaves únicas de llaves foráneas y relaciones uno a muchos. La multiplicidad de etapas es esperada y significa que la futura base a nivel viaje no debe enlazarlas directamente sin una regla de agregación aprobada. Los perfiles describen valores observados, no validan por sí solos la semántica ni la idoneidad de un campo para el modelo. El modo registrado describe la alternativa elegida; no proporciona automáticamente atributos de alternativas no elegidas.